# Importing CSV files into a database with SQLAlchemy Core

This lab imports a library's data, written in CSV files, into a SQLite database with SQLAlchemy Core: describe the tables of the books, the members and their loans, import the files, query the database, export the loans as CSV, then analyze them with pandas. It assumes the Core part of the "Getting started with SQLAlchemy" lab.

The steps build on each other: each solution defines what the next ones use. If you're stuck, run the solution before moving on.

## Setup

The next three cells write the library's data, one table per CSV file: the books (`books.csv`), the members (`members.csv`), and the dated loans (`loans.csv`), which name the book and the member by their title and their name; some of them are returned (they have a `returned` date). The one after creates the engine, on the `library.db` file.

In [ ]:
%%writefile books.csv
title,author
Vingt Mille Lieues sous les mers,Jules Verne
Voyage au centre de la Terre,Jules Verne
De la Terre à la Lune,Jules Verne
Le Tour du monde en quatre-vingts jours,Jules Verne
Michel Strogoff,Jules Verne
The Hound of the Baskervilles,Arthur Conan Doyle
"Arsène Lupin, gentleman-cambrioleur",Maurice Leblanc

In [ ]:
%%writefile members.csv
name
John Doe
Jane Doe
Sherlock Holmes
Arsène Lupin

In [ ]:
%%writefile loans.csv
book,member,date,returned
Voyage au centre de la Terre,Jane Doe,2026-01-03,2026-01-24
Le Tour du monde en quatre-vingts jours,Sherlock Holmes,2026-01-05,2026-01-19
Michel Strogoff,Sherlock Holmes,2026-01-07,
"Arsène Lupin, gentleman-cambrioleur",Jane Doe,2026-01-20,2026-02-01
The Hound of the Baskervilles,Sherlock Holmes,2026-02-02,
Voyage au centre de la Terre,John Doe,2026-02-10,
Le Tour du monde en quatre-vingts jours,Sherlock Holmes,2026-02-20,2026-03-06

In [ ]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///library.db")

## 1. Describe the tables

A member borrows several books, and a book is borrowed by several members over time: the loans table links the two with two foreign keys, and also holds its own information, the loan date and the return date. With a `MetaData` object named `metadata`, describe three tables:

- `books` (variable `table_books`): `id`, integer primary key; `title`, required and unique string; `author`, required string;
- `members` (variable `table_members`): `id`; `name`, required and unique string;
- `loans` (variable `table_loans`): `id`; `book_id` and `member_id`, required, foreign keys to the `id` columns of `books` and `members`; `loan_date`, required date; `return_date`, optional date (a current loan has none).

Drop the tables a previous run may have left, then create these. Why does a loan have its own `id`, rather than a primary key made of `book_id` and `member_id`?

**Useful documentation**

- [Constraints](https://docs.sqlalchemy.org/en/21/tutorial/metadata.html#declaring-simple-constraints)
- [`Date`](https://docs.sqlalchemy.org/en/21/core/type_basics.html#sqlalchemy.types.Date)

In [ ]:
# Your code here

### Additional hints

"Required" is `nullable=False`, "unique" `unique=True`, and a `Date` column takes `datetime.date` objects. A column that holds a `ForeignKey("books.id")` takes its type from it: no need to repeat it. `metadata.drop_all(engine)` then `metadata.create_all(engine)` drop then create all the tables described.

```python
from sqlalchemy import Column, Date, ForeignKey, Integer, MetaData, String, Table

metadata = MetaData()

table_books = Table(
  "books",
  metadata,
  Column("id", Integer, primary_key=True),
  Column("title", String, nullable=False, unique=True),
  Column("author", String, nullable=False),
)

table_members = Table(...)

table_loans = Table(
  "loans",
  metadata,
  Column("id", Integer, primary_key=True),
  Column("book_id", ForeignKey(...), nullable=False),
  ...,
)

metadata.drop_all(engine)
metadata.create_all(engine)
```

### Solution

In [ ]:
from sqlalchemy import Column, Date, ForeignKey, Integer, MetaData, String, Table

metadata = MetaData()

table_books = Table(
  "books",
  metadata,
  Column("id", Integer, primary_key=True),
  Column("title", String, nullable=False, unique=True),
  Column("author", String, nullable=False),
)

table_members = Table(
  "members",
  metadata,
  Column("id", Integer, primary_key=True),
  Column("name", String, nullable=False, unique=True),
)

table_loans = Table(
  "loans",
  metadata,
  Column("id", Integer, primary_key=True),
  Column("book_id", ForeignKey("books.id"), nullable=False),
  Column("member_id", ForeignKey("members.id"), nullable=False),
  Column("loan_date", Date, nullable=False),
  Column("return_date", Date),
)

metadata.drop_all(engine)
metadata.create_all(engine)

# The same member can borrow the same book several times (in the data,
# Sherlock Holmes and Le Tour du monde en quatre-vingts jours): the pair
# (book_id, member_id) doesn't identify a loan, so it can't be the primary
# key. Hence an id of its own for each loan.

## 2. Import the CSV files

Complete the `import_csv(directory, conn)` function, which reads the `books.csv`, `members.csv` and `loans.csv` files of the `directory` folder, and inserts their rows into the three tables, one `insert` statement per table. The loans name books and members by their title and their name, whereas the `loans` table expects their `id`: read them back from the database once the books and the members are inserted. Then call it on the current directory (`Path(".")`), in a single transaction, and display the number of rows of each table.

**Useful documentation**

- [`csv.DictReader`](https://docs.python.org/3/library/csv.html#csv.DictReader)
- [`date.fromisoformat`](https://docs.python.org/3/library/datetime.html#datetime.date.fromisoformat)

In [ ]:
from pathlib import Path

from sqlalchemy import Connection


def import_csv(directory: Path, conn: Connection) -> None:
  """Insert the books, members and loans of the CSV files of directory."""
  ...

In [ ]:
# Your code here

### Additional hints

`csv.DictReader(f)` reads a CSV file row by row, each row as a dictionary whose keys are the column names of the header; open the file with `newline=""`, as the `csv` module asks. The title that contains a comma is quoted in the file: `csv` removes the quotes. The headers of `books.csv` and `members.csv` are the column names: their rows are inserted as they are, all at once, with `conn.execute(insert(table_books), rows)`. For the loans, `dict(conn.execute(select(table_books.c.title, table_books.c.id)).all())` maps each title to its `id`. All the values read are strings: `date.fromisoformat` converts the dates, and `returned` is an empty string for a current loan. You should get 7 books, 4 members and 7 loans. Run again, the import raises an `IntegrityError`, since titles and names are unique: run the creation of the tables first.

```python
import csv
from datetime import date

from sqlalchemy import insert, select


def read_csv(path: Path) -> list[dict[str, str]]:
  with open(path, newline="", encoding="utf-8") as f:
    return list(csv.DictReader(f))


def read_date(text: str) -> date | None:
  return date.fromisoformat(text) if text else None


def import_csv(directory: Path, conn: Connection) -> None:
  conn.execute(insert(table_books), read_csv(directory / "books.csv"))
  conn.execute(...)
  book_ids = dict(conn.execute(select(table_books.c.title, table_books.c.id)).all())
  member_ids = ...
  conn.execute(
    insert(table_loans),
    [
      {
        "book_id": book_ids[row["book"]],
        "member_id": ...,
        "loan_date": ...,
        "return_date": ...,
      }
      for row in read_csv(directory / "loans.csv")
    ],
  )


with engine.begin() as conn:
  import_csv(Path("."), conn)

with engine.connect() as conn:
  for table in (table_books, table_members, table_loans):
    print(table.name, len(conn.execute(select(table)).all()))
```

### Solution

In [ ]:
import csv
from datetime import date

from sqlalchemy import insert, select


def read_csv(path: Path) -> list[dict[str, str]]:
  with open(path, newline="", encoding="utf-8") as f:
    return list(csv.DictReader(f))


def read_date(text: str) -> date | None:
  return date.fromisoformat(text) if text else None


def import_csv(directory: Path, conn: Connection) -> None:
  conn.execute(insert(table_books), read_csv(directory / "books.csv"))
  conn.execute(insert(table_members), read_csv(directory / "members.csv"))
  book_ids = dict(conn.execute(select(table_books.c.title, table_books.c.id)).all())
  member_ids = dict(
    conn.execute(select(table_members.c.name, table_members.c.id)).all()
  )
  conn.execute(
    insert(table_loans),
    [
      {
        "book_id": book_ids[row["book"]],
        "member_id": member_ids[row["member"]],
        "loan_date": read_date(row["date"]),
        "return_date": read_date(row["returned"]),
      }
      for row in read_csv(directory / "loans.csv")
    ],
  )


with engine.begin() as conn:
  import_csv(Path("."), conn)

with engine.connect() as conn:
  for table in (table_books, table_members, table_loans):
    print(table.name, len(conn.execute(select(table)).all()))

# For the books and the members, pandas would have done:
# pd.read_csv("books.csv").to_sql("books", engine, if_exists="append", index=False)
# The loans, though, require finding the ids of the books and the members.

## 3. Query the database

Each query of this part is written with `select`, and runs in a `with engine.connect() as conn:` block.

### The books borrowed by Sherlock Holmes

Display the title of the books Sherlock Holmes borrowed, with the date of each loan, sorted by date.

**Useful documentation**

- [Joins](https://docs.sqlalchemy.org/en/21/tutorial/data_select.html#explicit-from-clauses-and-joins)

In [ ]:
# Your code here

#### Additional hints

Start from the loans table, `select_from(table_loans)`, then add each other table with a `join(...)`: SQLAlchemy finds the join condition from the foreign key. Filter on `table_members.c.name` and sort by `table_loans.c.loan_date` with `order_by`. Each row of the result has one attribute per column of the `select`. You should get four loans, two of them of the same book, from January 5 to February 20, 2026.

```python
with engine.connect() as conn:
  query = (
    select(table_loans.c.loan_date, table_books.c.title)
    .select_from(table_loans)
    .join(table_books)
    .join(...)
    .where(...)
    .order_by(...)
  )
  for row in conn.execute(query):
    print(row.loan_date, row.title)
```

#### Solution

In [ ]:
with engine.connect() as conn:
  query = (
    select(table_loans.c.loan_date, table_books.c.title)
    .select_from(table_loans)
    .join(table_books)
    .join(table_members)
    .where(table_members.c.name == "Sherlock Holmes")
    .order_by(table_loans.c.loan_date)
  )
  for row in conn.execute(query):
    print(row.loan_date, row.title)

### The current loans

Display the current loans, those with no return date, by loan date: the date, the title of the book and the name of the member.

**Useful documentation**

- [`is_`](https://docs.sqlalchemy.org/en/21/core/sqlelement.html#sqlalchemy.sql.expression.ColumnOperators.is_)

In [ ]:
# Your code here

#### Additional hints

In SQL, a missing value is tested with `IS NULL`, which SQLAlchemy writes `table_loans.c.return_date.is_(None)` (rather than `== None`, which style checkers flag). The query joins the three tables, like the previous one. You should get 3 loans.

```python
with engine.connect() as conn:
  current = ...
  query = (
    select(table_loans.c.loan_date, table_books.c.title, table_members.c.name)
    .select_from(table_loans)
    .join(...)
    .join(...)
    .where(...)
    .order_by(...)
  )
  for row in conn.execute(query):
    print(row.loan_date, row.title, row.name)
```

#### Solution

In [ ]:
with engine.connect() as conn:
  current = table_loans.c.return_date.is_(None)
  query = (
    select(table_loans.c.loan_date, table_books.c.title, table_members.c.name)
    .select_from(table_loans)
    .join(table_books)
    .join(table_members)
    .where(current)
    .order_by(table_loans.c.loan_date)
  )
  for row in conn.execute(query):
    print(row.loan_date, row.title, row.name)

### The available books

Display, in alphabetical order, the title of the available books: those with no current loan.

**Useful documentation**

- [`not_in`](https://docs.sqlalchemy.org/en/21/core/sqlelement.html#sqlalchemy.sql.expression.ColumnOperators.not_in)

In [ ]:
# Your code here

#### Additional hints

`column.not_in(subquery)` keeps the rows whose value isn't in the result of another `select` (in SQL, `NOT IN (SELECT ...)`): here, the books whose `id` isn't among the `book_id` of the current loans. The "current" condition is the one of the previous exercise. Careful: a returned book may have been borrowed again. `conn.scalars(...)` returns the first column of each row. You should get 4 books.

```python
with engine.connect() as conn:
  current = table_loans.c.return_date.is_(None)
  borrowed = select(...).where(current)
  query = select(table_books.c.title).where(...).order_by(table_books.c.title)
  for title in conn.scalars(query):
    print(title)
```

#### Solution

In [ ]:
with engine.connect() as conn:
  current = table_loans.c.return_date.is_(None)
  borrowed = select(table_loans.c.book_id).where(current)
  query = (
    select(table_books.c.title)
    .where(table_books.c.id.not_in(borrowed))
    .order_by(table_books.c.title)
  )
  for title in conn.scalars(query):
    print(title)

# Voyage au centre de la Terre isn't there: returned by Jane Doe, it was
# borrowed again, by John Doe. A book is available if it has no current loan,
# not if it has a returned loan.

## 4. Export the loans as CSV

Complete the `export_csv(path, conn)` function, which writes all the loans, by loan date, into the CSV file `path`: one row per loan, with the columns `book`, `member`, `loan_date` and `return_date` (empty for a current loan). Call it to write `export.csv`, then display the content of the file.

**Useful documentation**

- [`csv.DictWriter`](https://docs.python.org/3/library/csv.html#csv.DictWriter)
- [`ColumnElement.label`](https://docs.sqlalchemy.org/en/21/core/sqlelement.html#sqlalchemy.sql.expression.ColumnElement.label)
- [`Result.mappings`](https://docs.sqlalchemy.org/en/21/core/connections.html#sqlalchemy.engine.Result.mappings)

In [ ]:
def export_csv(path: str, conn: Connection) -> None:
  """Write the loans of the database into the CSV file path."""
  ...

In [ ]:
# Your code here

### Additional hints

`.label("book")` names a column of the `select`, and `conn.execute(query).mappings()` returns each row as a dictionary whose keys are the column names. `csv.DictWriter(f, fieldnames=[...])` writes such dictionaries: `writeheader()` writes the header, then `writerow(...)` each row; open the file with `newline=""`. `csv` writes a date in the `2026-01-03` format, and `None` as an empty field. The file should have 8 lines: the header and 7 loans.

```python
def export_csv(path: str, conn: Connection) -> None:
  query = (
    select(
      table_books.c.title.label("book"),
      ...,
    )
    .select_from(table_loans)
    .join(...)
    .join(...)
    .order_by(...)
  )
  with open(path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
      f, fieldnames=["book", "member", "loan_date", "return_date"]
    )
    writer.writeheader()
    for row in conn.execute(query).mappings():
      ...


with engine.connect() as conn:
  export_csv("export.csv", conn)

with open("export.csv", encoding="utf-8") as f:
  print(f.read())
```

### Solution

In [ ]:
def export_csv(path: str, conn: Connection) -> None:
  query = (
    select(
      table_books.c.title.label("book"),
      table_members.c.name.label("member"),
      table_loans.c.loan_date,
      table_loans.c.return_date,
    )
    .select_from(table_loans)
    .join(table_books)
    .join(table_members)
    .order_by(table_loans.c.loan_date)
  )
  with open(path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
      f, fieldnames=["book", "member", "loan_date", "return_date"]
    )
    writer.writeheader()
    for row in conn.execute(query).mappings():
      writer.writerow(row)


with engine.connect() as conn:
  export_csv("export.csv", conn)

with open("export.csv", encoding="utf-8") as f:
  print(f.read())

## 5. Analyze the loans with pandas

With `pd.read_sql`, load into a `DataFrame` `loans` the result of a `select` query that gives, for each loan, the name of the member (column `member`), the title of the book (column `book`) and the loan date. Then display the number of loans of each member, from the largest to the smallest.

**Useful documentation**

- [`Series.value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)

In [ ]:
# Your code here

### Additional hints

`pd.read_sql(query, engine)` runs the query and returns a `DataFrame`, one column per element of the `select`, named by its `label`. The query joins the three tables, like the export's. `value_counts()` counts the values of a column, from the most frequent to the least frequent. You should get Sherlock Holmes 4, Jane Doe 2 and John Doe 1.

```python
import pandas as pd

query = (
  select(table_members.c.name.label("member"), ..., table_loans.c.loan_date)
  .select_from(table_loans)
  .join(...)
  .join(...)
)
loans = pd.read_sql(query, engine)
print(...)
```

### Solution

In [ ]:
import pandas as pd

query = (
  select(
    table_members.c.name.label("member"),
    table_books.c.title.label("book"),
    table_loans.c.loan_date,
  )
  .select_from(table_loans)
  .join(table_members)
  .join(table_books)
)
loans = pd.read_sql(query, engine)
print(loans["member"].value_counts())